Wczytanie danych

In [83]:
import os, json, time, numpy as np, pandas as pd, dash
import plotly.express as px, plotly.graph_objs as go, dash_bootstrap_components as dbc
from dash import Input, Output, State, dcc, html, dash_table, ALL, MATCH, ctx
#region Start
app = dash.Dash(
    __name__,
    external_stylesheets=[
        dbc.themes.BOOTSTRAP, #Wygląd
        "https://cdn.jsdelivr.net/npm/bootstrap-icons@1.11.0/font/bootstrap-icons.css"
    ],
    suppress_callback_exceptions=True
)
DATA_FOLDER = 'Dane Dash/' #Folder do zczytania gotowych danych
frames = []  
for filename in os.listdir(DATA_FOLDER):
    if filename.endswith('.json'):
        filepath = os.path.join(DATA_FOLDER, filename)
        frames.append(pd.read_json(filepath))
df = pd.concat(frames, ignore_index=True)

Kolumny z dniem tygodnia i godziną

In [301]:
# Zmiana na czas polski
timestamp = df['timestamp']
timestamp_pl = timestamp.dt.tz_convert('Europe/Warsaw')
df['timestamp_pl'] = timestamp_pl
df['day_of_year'] = timestamp_pl.dt.day_of_year
df['weekday'] = timestamp_pl.dt.day_name('Pl')
df['weekday_number'] = timestamp_pl.dt.weekday + 1
df['hour'] = timestamp_pl.dt.hour
df.sample(2)

,timestamp,duration,app,title,status,category,osoba,device,timestamp_pl,weekday,weekday_number,hour,day_of_year
26408,2025-12-27 13:43:55.960000+00:00,5.045,Obsidian.exe,Widok grafu - Obsidian Vault - Obsidian v1.10.6,not-afk,gry komputerowe,A,PC,2025-12-27 14:43:55.960000+01:00,Sobota,6,14,361
21247,2025-12-22 11:52:45.050000+00:00,2.030,MATLAB.exe,MATLAB App,not-afk,studia,A,PC,2025-12-22 12:52:45.050000+01:00,Poniedziałek,1,12,356


# Średnia aktywność w dniach tygodnia

In [352]:
# po czym agregować kategoria / aplikacja
agregator = "category"
agregator = "app"

In [353]:
Dni = df[['osoba',"weekday","weekday_number","day_of_year"]].drop_duplicates()
Dni = Dni.groupby(["osoba","weekday","weekday_number"])['day_of_year'].aggregate(n = 'count').reset_index().sort_values(['osoba','weekday_number'])
Dni

,osoba,weekday,weekday_number,n
3,A,Poniedziałek,1,3
5,A,Wtorek,2,3
6,A,Środa,3,3
0,A,Czwartek,4,3
2,A,Piątek,5,3
4,A,Sobota,6,4
1,A,Niedziela,7,3
10,AK,Poniedziałek,1,4
12,AK,Wtorek,2,3
13,AK,Środa,3,3


In [354]:
df2 = df.groupby(["osoba","weekday","weekday_number",agregator])['duration'].aggregate(total_activity = 'sum').reset_index()
df2['total_activity'] /= 3600
# Teraz podzilić przez liczbę zarejestrowanych dni
df2 = pd.merge(df2,Dni)
df2['average_activity'] = df2['total_activity'] / df2['n']
df2

,osoba,weekday,weekday_number,app,total_activity,n,average_activity
0,A,Czwartek,4,ApplicationFrameHost.exe,0.010690,3,0.003563
1,A,Czwartek,4,Discord.exe,0.020874,3,0.006958
2,A,Czwartek,4,Foundry.exe,0.807427,3,0.269142
3,A,Czwartek,4,League of Legends.exe,8.579881,3,2.859960
4,A,Czwartek,4,LeagueClient.exe,0.001850,3,0.000617
...,...,...,...,...,...,...,...
339,AK,Środa,3,notepad.exe,0.015485,3,0.005162
340,Jan,Niedziela,7,steamwebhelper.exe,0.003946,1,0.003946
341,Jan,Piątek,5,Obsidian.exe,0.000280,1,0.000280
342,Jeremiasz,Czwartek,4,msedge.exe,0.000568,1,0.000568


In [355]:
# To będzie do czegoś innego
Suma_z_dnia = df.groupby(['osoba','day_of_year'])['duration'].sum().reset_index().sort_values('duration',ascending=False)
Suma_z_dnia['duration'] /= 3600
Suma_z_dnia

,osoba,day_of_year,duration
5,A,349,14.402186
16,A,360,12.646857
17,A,361,12.493067
43,AK,365,12.306290
32,AK,348,12.105830
0,A,1,11.561803
13,A,357,11.257502
22,AK,1,10.936024
18,A,362,10.879101
15,A,359,10.692861


In [356]:
osoba = "A"
df3 = df2[df2['osoba'] == osoba]
ylim = np.ceil(df2.groupby(['osoba','weekday'])['average_activity'].sum().reset_index()['average_activity'].max())
px.bar(df3,
       x='weekday',
       y = 'average_activity',
       color=agregator,
       category_orders={'weekday':['Poniedziałek','Wtorek','Środa','Czwartek','Piątek','Sobota','Niedziela']},
       labels={'weekday_number':'dzień tygodnia','average_activity':'średnia aktywność (godz.)',agregator:agregator},
       title=f"Średnia aktywność w dniach tygodnia: {osoba}",
       template='plotly_white',
       range_y=[0,ylim])